In [1]:
import os
import numpy as np
import pandas as pd

from sklearn.model_selection import train_test_split
import warnings
warnings.simplefilter("ignore")

In [2]:
project_dir=r"C:\Users\koush\anaconda3\flights-sagemaker"
data_dir="data"

In [3]:
data=os.path.join(project_dir,data_dir,'flight_prices.xlsx')
df=pd.read_excel(data)

In [4]:
df.head()

,Airline,Date_of_Journey,Source,Destination,Route,Dep_Time,Arrival_Time,Duration,Total_Stops,Additional_Info,Price
0,IndiGo,24/03/2019,Banglore,New Delhi,BLR → DEL,22:20,01:10 22 Mar,2h 50m,non-stop,No info,3897
1,Air India,1/05/2019,Kolkata,Banglore,CCU → IXR → BBI → BLR,05:50,13:15,7h 25m,2 stops,No info,7662
2,Jet Airways,9/06/2019,Delhi,Cochin,DEL → LKO → BOM → COK,09:25,04:25 10 Jun,19h,2 stops,No info,13882
3,IndiGo,12/05/2019,Kolkata,Banglore,CCU → NAG → BLR,18:05,23:30,5h 25m,1 stop,No info,6218
4,IndiGo,01/03/2019,Banglore,New Delhi,BLR → NAG → DEL,16:50,21:35,4h 45m,1 stop,No info,13302


In [5]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 10683 entries, 0 to 10682
Data columns (total 11 columns):
 #   Column           Non-Null Count  Dtype 
---  ------           --------------  ----- 
 0   Airline          10683 non-null  object
 1   Date_of_Journey  10683 non-null  object
 2   Source           10683 non-null  object
 3   Destination      10683 non-null  object
 4   Route            10682 non-null  object
 5   Dep_Time         10683 non-null  object
 6   Arrival_Time     10683 non-null  object
 7   Duration         10683 non-null  object
 8   Total_Stops      10682 non-null  object
 9   Additional_Info  10683 non-null  object
 10  Price            10683 non-null  int64 
dtypes: int64(1), object(10)
memory usage: 918.2+ KB


- The dataset contains 10683 rows and 11 features
- Column 'Route' and 'Total_Stops' have missing value each
- Dtype for some features is inappropriate

## Preliminary Analysis

#### Check data types

In [6]:
df.head()

,Airline,Date_of_Journey,Source,Destination,Route,Dep_Time,Arrival_Time,Duration,Total_Stops,Additional_Info,Price
0,IndiGo,24/03/2019,Banglore,New Delhi,BLR → DEL,22:20,01:10 22 Mar,2h 50m,non-stop,No info,3897
1,Air India,1/05/2019,Kolkata,Banglore,CCU → IXR → BBI → BLR,05:50,13:15,7h 25m,2 stops,No info,7662
2,Jet Airways,9/06/2019,Delhi,Cochin,DEL → LKO → BOM → COK,09:25,04:25 10 Jun,19h,2 stops,No info,13882
3,IndiGo,12/05/2019,Kolkata,Banglore,CCU → NAG → BLR,18:05,23:30,5h 25m,1 stop,No info,6218
4,IndiGo,01/03/2019,Banglore,New Delhi,BLR → NAG → DEL,16:50,21:35,4h 45m,1 stop,No info,13302


In [7]:
df.dtypes

Airline            object
Date_of_Journey    object
Source             object
Destination        object
Route              object
Dep_Time           object
Arrival_Time       object
Duration           object
Total_Stops        object
Additional_Info    object
Price               int64
dtype: object

#### Check for duplicates

In [8]:
df.duplicated().sum()
#df.shape

220

In [9]:
(
df.loc[df.duplicated(keep=False)].sort_values(['Airline','Date_of_Journey','Source','Destination'])
)

,Airline,Date_of_Journey,Source,Destination,Route,Dep_Time,Arrival_Time,Duration,Total_Stops,Additional_Info,Price
6321,Air India,01/03/2019,Banglore,New Delhi,BLR → BOM → AMD → DEL,08:50,23:55 02 Mar,39h 5m,2 stops,No info,17135
9848,Air India,01/03/2019,Banglore,New Delhi,BLR → BOM → AMD → DEL,08:50,23:55 02 Mar,39h 5m,2 stops,No info,17135
572,Air India,03/03/2019,Banglore,New Delhi,BLR → DEL,21:10,23:55,2h 45m,non-stop,No info,7591
8168,Air India,03/03/2019,Banglore,New Delhi,BLR → DEL,21:10,23:55,2h 45m,non-stop,No info,7591
1495,Air India,1/04/2019,Kolkata,Banglore,CCU → DEL → COK → BLR,10:00,01:20 02 Apr,15h 20m,2 stops,No info,10408
...,...,...,...,...,...,...,...,...,...,...,...
2692,SpiceJet,24/03/2019,Banglore,New Delhi,BLR → DEL,05:45,08:35,2h 50m,non-stop,No check-in baggage included,4273
2870,SpiceJet,24/03/2019,Banglore,New Delhi,BLR → DEL,05:45,08:35,2h 50m,non-stop,No check-in baggage included,4273
3711,SpiceJet,24/03/2019,Banglore,New Delhi,BLR → DEL,20:30,23:20,2h 50m,non-stop,No check-in baggage included,3873
2634,Vistara,24/03/2019,Banglore,New Delhi,BLR → DEL,11:30,14:10,2h 40m,non-stop,No info,5403


#### Detailed Analysis

###### Airline

In [10]:
df.Airline.unique()

array(['IndiGo', 'Air India', 'Jet Airways', 'SpiceJet',
       'Multiple carriers', 'GoAir', 'Vistara', 'Air Asia',
       'Vistara Premium economy', 'Jet Airways Business',
       'Multiple carriers Premium economy', 'Trujet'], dtype=object)

- Some of the entries have inconsistent/inaccurate values

In [48]:
(
    df
    .Airline
    .str.replace(" Premium economy","")
    .str.replace(" Business","")
    .str.title() #first letter of a word is capital
    .unique()
)

array(['Indigo', 'Air India', 'Jet Airways', 'Spicejet',
       'Multiple Carriers', 'Goair', 'Vistara', 'Air Asia', 'Trujet'],
      dtype=object)

##### Date of Journey

In [14]:
pd.to_datetime(df.Date_of_Journey,dayfirst=True)

0       2019-03-24
1       2019-05-01
2       2019-06-09
3       2019-05-12
4       2019-03-01
           ...    
10678   2019-04-09
10679   2019-04-27
10680   2019-04-27
10681   2019-03-01
10682   2019-05-09
Name: Date_of_Journey, Length: 10683, dtype: datetime64[ns]

###### Sources

In [15]:
df.Source.unique()

array(['Banglore', 'Kolkata', 'Delhi', 'Chennai', 'Mumbai'], dtype=object)

###### Destination

In [16]:
df.Destination.unique()

array(['New Delhi', 'Banglore', 'Cochin', 'Kolkata', 'Delhi', 'Hyderabad'],
      dtype=object)

-Route -- since this is not needed for any info.

##### Dep Time

In [17]:
df.Dep_Time

0        22:20
1        05:50
2        09:25
3        18:05
4        16:50
         ...  
10678    19:55
10679    20:45
10680    08:20
10681    11:30
10682    10:55
Name: Dep_Time, Length: 10683, dtype: object

In [18]:
#checking whether any unnecessary character present in the data or not
(
df
    .Dep_Time
    .loc[lambda ser: ser.str.contains("[^0-9:]")]
)

Series([], Name: Dep_Time, dtype: object)

##### Arrival time

In [19]:
df.Arrival_Time

0        01:10 22 Mar
1               13:15
2        04:25 10 Jun
3               23:30
4               21:35
             ...     
10678           22:25
10679           23:20
10680           11:20
10681           14:10
10682           19:15
Name: Arrival_Time, Length: 10683, dtype: object

In [20]:
#checking whether any unnecessary character present in the data or not and handling it (just to know about the feature)
(
df
    .Arrival_Time
    .loc[lambda ser: ser.str.contains("[^0-9:]")]
    .str.split(" ",n=1) #n for no. of splits
    .str.get(0)
)

0        01:10
2        04:25
6        10:25
7        05:05
8        10:25
         ...  
10666    19:00
10667    20:20
10672    19:00
10673    04:25
10674    21:20
Name: Arrival_Time, Length: 4335, dtype: object

In [21]:
pd.to_datetime(df.Arrival_Time).dt.time

0        01:10:00
1        13:15:00
2        04:25:00
3        23:30:00
4        21:35:00
           ...   
10678    22:25:00
10679    23:20:00
10680    11:20:00
10681    14:10:00
10682    19:15:00
Name: Arrival_Time, Length: 10683, dtype: object

##### Duration

In [22]:
df.Duration

0        2h 50m
1        7h 25m
2           19h
3        5h 25m
4        4h 45m
          ...  
10678    2h 30m
10679    2h 35m
10680        3h
10681    2h 40m
10682    8h 20m
Name: Duration, Length: 10683, dtype: object

In [23]:
df.Duration.str.split()

0        [2h, 50m]
1        [7h, 25m]
2            [19h]
3        [5h, 25m]
4        [4h, 45m]
           ...    
10678    [2h, 30m]
10679    [2h, 35m]
10680         [3h]
10681    [2h, 40m]
10682    [8h, 20m]
Name: Duration, Length: 10683, dtype: object

In [24]:
(  df
    .Duration
    .loc[lambda ser: ~ser.str.contains('m')] #duration which doesn't contains m
    .unique()
)

array(['19h', '23h', '22h', '12h', '3h', '5h', '10h', '18h', '24h', '15h',
       '16h', '8h', '14h', '20h', '13h', '11h', '9h', '27h', '26h', '4h',
       '7h', '30h', '21h', '28h', '47h', '6h', '25h', '38h', '34h'],
      dtype=object)

In [25]:
(  df
    .Duration
    .loc[lambda ser: ~ser.str.contains('h')] #duration which doesn't contains h
    #.unique()
)

6474    5m
Name: Duration, dtype: object

In [26]:
df.iloc[[6474]]

,Airline,Date_of_Journey,Source,Destination,Route,Dep_Time,Arrival_Time,Duration,Total_Stops,Additional_Info,Price
6474,Air India,6/03/2019,Mumbai,Hyderabad,BOM → GOI → PNQ → HYD,16:50,16:55,5m,2 stops,No info,17327


- Clearly the duration of 5 minutes is invalid, so will delete it

In [27]:
#for validation whether the conversion is correct or not
(
df
    .Duration
    .drop(index=[6474])
    .str.split(" ",expand=True) #expand=true returing dataframe
    .set_axis(['hour','minute'],axis=1)
    .assign(
        hour=lambda df_:(
        df_
        .hour
        .str.replace("h","")
        .astype(int)
        .mul(60) #converting to minutes
        ),
        minute=lambda df_:(
        df_
        .minute
        .str.replace("m",""))
        .fillna("0") #fill missing values as 0(here still minute is str)
        .astype(int)
    )
    #.dtypes
    #.isna().sum() #missing value in hour and minute column
    .sum(axis=1)
    .rename("duration_min")
    .to_frame() #to dataframe
    .join(df.Duration)
    .head()
)

,duration_min,Duration
0,170,2h 50m
1,445,7h 25m
2,1140,19h
3,325,5h 25m
4,285,4h 45m


##### Total_Stops

In [28]:
df.Total_Stops.unique()

array(['non-stop', '2 stops', '1 stop', '3 stops', nan, '4 stops'],
      dtype=object)

In [29]:
(
    df
    .Total_Stops
    .replace("non-stop","0")
    .str.replace("stops?","",regex=True) #"stops?" means it will either search for stop or stops
    .pipe(lambda ser: pd.to_numeric(ser)) #int cant work due to nan so float is only compatible type
)

0        0.0
1        2.0
2        2.0
3        1.0
4        1.0
        ... 
10678    0.0
10679    0.0
10680    0.0
10681    0.0
10682    2.0
Name: Total_Stops, Length: 10683, dtype: float64

##### Additional Info

In [30]:
df.Additional_Info.unique() #No info, No Info are same so this has to be corrected

array(['No info', 'In-flight meal not included',
       'No check-in baggage included', '1 Short layover', 'No Info',
       '1 Long layover', 'Change airports', 'Business class',
       'Red-eye flight', '2 Long layover'], dtype=object)

### Cleaning operations

In [31]:
def convert_to_minutes(df):
    return(
            df
            .str.split(" ",expand=True) #expand=true returing dataframe
            .set_axis(['hour','minute'],axis=1)
            .assign(
            hour=lambda df_:(
            df_
            .hour
            .str.replace("h","")
            .astype(int)
            .mul(60) #converting to minutes
            ),
            minute=lambda df_:(
            df_
            .minute
            .str.replace("m",""))
            .fillna("0") #fill missing values as 0(here still minute is str)
            .astype(int)
        )
         #.dtypes
    #.isna().sum() #missing value in hour and minute column
        .sum(axis=1)
    )

In [49]:
def clean_data(df):
    return(
    df
        .drop(index=[6474])
        .drop_duplicates()
        .assign(**{
            col: df[col].str.strip()
            for col in df.select_dtypes(include='O').columns
        })
        .rename(columns=str.lower)
        .assign(
        airline=lambda df_: (
    df_
    .airline
    .str.replace(" Premium economy","")
    .str.replace(" Business","")
    .str.title() #first letter of a word is capital
    ),
    date_of_journey=lambda df_: pd.to_datetime(df_.date_of_journey,dayfirst=True),
            dep_time= lambda df_: pd.to_datetime(df_.dep_time).dt.time,
            arrival_time=lambda df_:pd.to_datetime(df_.arrival_time).dt.time,
            duration=lambda df_: df_.duration.pipe(convert_to_minutes),
            total_stops=lambda df_: (
            df_
            .total_stops
            .replace("non-stop","0")
            .str.replace("stops?","",regex=True) #"stops?" means it will either search for stop or stops
            .pipe(lambda ser: pd.to_numeric(ser)  #int cant work due to nan so float is only compatible type
                 )),
            additional_info=lambda df_: df_.additional_info.replace('No info',"No Info")
    )
    .drop(columns='route')
)
        

In [50]:
df_cleaned=clean_data(df)

In [51]:
df_cleaned

,airline,date_of_journey,source,destination,dep_time,arrival_time,duration,total_stops,additional_info,price
0,Indigo,2019-03-24,Banglore,New Delhi,22:20:00,01:10:00,170,0.0,No Info,3897
1,Air India,2019-05-01,Kolkata,Banglore,05:50:00,13:15:00,445,2.0,No Info,7662
2,Jet Airways,2019-06-09,Delhi,Cochin,09:25:00,04:25:00,1140,2.0,No Info,13882
3,Indigo,2019-05-12,Kolkata,Banglore,18:05:00,23:30:00,325,1.0,No Info,6218
4,Indigo,2019-03-01,Banglore,New Delhi,16:50:00,21:35:00,285,1.0,No Info,13302
...,...,...,...,...,...,...,...,...,...,...
10678,Air Asia,2019-04-09,Kolkata,Banglore,19:55:00,22:25:00,150,0.0,No Info,4107
10679,Air India,2019-04-27,Kolkata,Banglore,20:45:00,23:20:00,155,0.0,No Info,4145
10680,Jet Airways,2019-04-27,Banglore,Delhi,08:20:00,11:20:00,180,0.0,No Info,7229
10681,Vistara,2019-03-01,Banglore,New Delhi,11:30:00,14:10:00,160,0.0,No Info,12648


### Split the data

In [52]:
df_final = df_cleaned.sample(1000) #sample size doesn't matter

In [53]:
X = df_final.drop(columns="price")
y = df_final.price.copy()

In [54]:
X_, X_test, y_, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
X_train, X_val, y_train, y_val = train_test_split(X_, y_, test_size=0.2, random_state=42)

print(X_train.shape, y_train.shape)
print(X_val.shape, y_val.shape)
print(X_test.shape, y_test.shape)

(640, 9) (640,)
(160, 9) (160,)
(200, 9) (200,)


### Export the Subsets

In [55]:
def export_data(X, y, name):
	file_name = f"{name}.csv"
	file_path = os.path.join(project_dir, data_dir, file_name)

	X.join(y).to_csv(file_path, index=False) 

	return pd.read_csv(file_path).head()

In [56]:
export_data(X_train, y_train, "train")

,airline,date_of_journey,source,destination,dep_time,arrival_time,duration,total_stops,additional_info,price
0,Jet Airways,2019-04-01,Kolkata,Banglore,06:30:00,16:20:00,590,1.0,No Info,12996
1,Indigo,2019-06-06,Delhi,Cochin,14:20:00,22:30:00,490,1.0,No Info,6938
2,Indigo,2019-05-06,Kolkata,Banglore,15:15:00,17:45:00,150,0.0,No Info,4804
3,Air India,2019-03-01,Chennai,Kolkata,11:40:00,13:55:00,135,0.0,No Info,19630
4,Indigo,2019-05-21,Banglore,Delhi,21:15:00,00:15:00,180,0.0,No Info,3943


In [57]:
export_data(X_val, y_val, "val")

,airline,date_of_journey,source,destination,dep_time,arrival_time,duration,total_stops,additional_info,price
0,Multiple Carriers,2019-06-27,Delhi,Cochin,09:15:00,19:00:00,585,1.0,No Info,11622
1,Multiple Carriers,2019-03-06,Delhi,Cochin,09:45:00,16:10:00,385,1.0,No Info,10276
2,Spicejet,2019-05-01,Kolkata,Banglore,17:10:00,19:40:00,150,0.0,No Info,4174
3,Jet Airways,2019-05-01,Banglore,Delhi,11:10:00,14:05:00,175,0.0,No Info,7229
4,Goair,2019-05-03,Banglore,Delhi,20:55:00,23:40:00,165,0.0,No Info,4239


In [58]:
export_data(X_test, y_test, "test")

,airline,date_of_journey,source,destination,dep_time,arrival_time,duration,total_stops,additional_info,price
0,Indigo,2019-05-24,Banglore,Delhi,07:10:00,10:05:00,175,0.0,No Info,4823
1,Indigo,2019-03-21,Delhi,Cochin,07:45:00,13:40:00,355,1.0,No Info,6101
2,Jet Airways,2019-05-24,Kolkata,Banglore,20:00:00,23:35:00,1655,1.0,In-flight meal not included,10844
3,Air India,2019-05-21,Kolkata,Banglore,16:50:00,13:45:00,1255,2.0,No Info,13484
4,Air India,2019-06-09,Delhi,Cochin,19:45:00,19:15:00,1410,2.0,No Info,9968


In [59]:
df_cleaned

,airline,date_of_journey,source,destination,dep_time,arrival_time,duration,total_stops,additional_info,price
0,Indigo,2019-03-24,Banglore,New Delhi,22:20:00,01:10:00,170,0.0,No Info,3897
1,Air India,2019-05-01,Kolkata,Banglore,05:50:00,13:15:00,445,2.0,No Info,7662
2,Jet Airways,2019-06-09,Delhi,Cochin,09:25:00,04:25:00,1140,2.0,No Info,13882
3,Indigo,2019-05-12,Kolkata,Banglore,18:05:00,23:30:00,325,1.0,No Info,6218
4,Indigo,2019-03-01,Banglore,New Delhi,16:50:00,21:35:00,285,1.0,No Info,13302
...,...,...,...,...,...,...,...,...,...,...
10678,Air Asia,2019-04-09,Kolkata,Banglore,19:55:00,22:25:00,150,0.0,No Info,4107
10679,Air India,2019-04-27,Kolkata,Banglore,20:45:00,23:20:00,155,0.0,No Info,4145
10680,Jet Airways,2019-04-27,Banglore,Delhi,08:20:00,11:20:00,180,0.0,No Info,7229
10681,Vistara,2019-03-01,Banglore,New Delhi,11:30:00,14:10:00,160,0.0,No Info,12648
